In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Pinecone
import pinecone
import os
import pypdf


c:\Users\blaks\RAG\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [60]:
#document loader
def dcocument_loader(file_paths: list):
    all_pages=[]
    for file in file_paths:
        loader=PyPDFLoader(file)
        pages=loader.load_and_split()
        all_pages.extend(pages)
    return all_pages

In [61]:
result=dcocument_loader(['C:/Users/blaks/RAG/RAG/dataset/attention-is-all-you-need-Paper.pdf'])
print(result[0])

page_content='Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to trai

In [62]:
#text splitter
def text_splitter(all_pages: list):
    splitter=RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    return splitter.split_documents(all_pages)

In [63]:
splitter=text_splitter(result)
#print(splitter[0])

In [64]:
#generate embeddings
Enbedding_MODEL_NAME=os.getenv("Enbedding_MODEL_NAME")
def create_embeddings(chunks: list):
    Embed_model=HuggingFaceEmbeddings(model_name=Enbedding_MODEL_NAME)
    embeddings=Embed_model.embed_documents([chunk.page_content for chunk in chunks])
    return embeddings

In [23]:
embedding=create_embeddings(splitter)
#print(embedding[0])
print(f"Dimension: {len(embedding[0])}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8665.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimension: 384


In [65]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec

In [66]:
#create infdex in pinecone 
from dotenv import load_dotenv


load_dotenv()
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")
pc=Pinecone(api_key="pcsk_2XGyW7_9MvBNP3qLPXy5Do81S92HahZMQr1iomhBV5BqKsCwTRU3me392bZ4BQxGVyVH8w")

index_name="transformer-paper"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=len(embedding[0]),
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
        
    )
    index=pc.Index(index_name)
    

In [ ]:
vector_store=PineconeVectorStore(index=index, embedding=embed_model)

NameError: name 'index' is not defined